# Demo: orquestador multi-agente (LangGraph)

Este notebook muestra el **flujo de delegación** del orquestador: un *Supervisor* recibe la consulta, delega al **Investigador**, después al **Analista**, valida con una rúbrica y cierra con una **síntesis final**.

**Cómo usarlo en Google Colab**
1. Guardá tu clave de OpenAI en *Secretos* (ícono de la llave, a la izquierda) con el nombre `OPENAI_API_KEY` y activá "Acceso desde el notebook". Si no lo hacés, el notebook te la pide por teclado.
2. Ejecutá las celdas en orden (`Entorno de ejecución > Ejecutar todo`).

La clave nunca queda escrita en el notebook ni en el repositorio.

## 1. Traer el código e instalar dependencias

In [ ]:
# Cambiá esta URL por la de tu repositorio público en GitHub
REPO_URL = "https://github.com/TU_USUARIO/ejercicio-6-orquestador-multiagente"

import os
carpeta = REPO_URL.rstrip("/").split("/")[-1]
if not os.path.isdir(carpeta) and not os.path.exists("graph.py"):
    !git clone {REPO_URL}
if os.path.isdir(carpeta):
    %cd {carpeta}
!pip install -q -r requirements.txt

## 2. Configurar la clave de OpenAI

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Clave cargada desde los Secretos de Colab.")
except Exception:
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Pegá tu OPENAI_API_KEY: ")

## 3. El grafo

Topología jerárquica: todos los especialistas reportan al supervisor. Las flechas punteadas son aristas condicionales (el supervisor elige el siguiente nodo con un `Literal`).

In [ ]:
from graph import build_graph, create_llm

grafo = build_graph(create_llm())
try:
    from IPython.display import Image, display
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception:
    # Sin acceso al servicio que dibuja el PNG: mostramos el código Mermaid
    print(grafo.get_graph().draw_mermaid())

## 4. Ejecutar el flujo de delegación

La consulta exige dos dominios: **investigación** (reseñas y comisiones) y **análisis** (promedio, sentimiento y costo a 3 años). Se ve cada decisión del supervisor con su rúbrica, el aporte de cada especialista y la respuesta final.

In [ ]:
from main import PREGUNTA_DEMO, ejecutar

traza = await ejecutar(PREGUNTA_DEMO)

## 5. Resumen de la delegación

In [ ]:
print("Flujo:", " -> ".join(["supervisor"] + traza["flujo"]), "-> síntesis\n")

for c in traza["contribuciones"]:
    print(f"• {c['agente']:<13} (paso {c['paso']}) herramientas: {', '.join(c['herramientas'])}")

print("\nRúbrica del supervisor en cada decisión:")
for d in traza["decisiones"]:
    ev = d["evaluacion"]
    print(f"  decisión {d['paso']} -> {d['siguiente']:<12} "
          f"fuentes={ev['datos_con_fuente']!s:<5} cálculos={ev['calculos_hechos_por_analista']!s:<5} "
          f"responde_todo={ev['responde_toda_la_consulta']}")

## 6. Probá con tu propia consulta

Cambiá el texto y volvé a ejecutar la celda. Por ejemplo, compará los dos productos de la base (*Tarjeta Plus* y *Cuenta Sueldo Digital*).

In [ ]:
mi_pregunta = "Compará la Tarjeta Plus y la Cuenta Sueldo Digital: calificación promedio de cada una y cuál conviene según costo y opiniones."

traza2 = await ejecutar(mi_pregunta)